# 📊 K-Means Clustering — จัดกลุ่ม มอก. จากขอบข่าย (TF-IDF)

**Goal:** จัดกลุ่ม 148 มาตรฐาน มอก. ตามความคล้ายคลึงของข้อความขอบข่าย

**Pipeline:** TF-IDF Vectorization → StandardScaler → KMeans → PCA 2D

**Output:**
- PCA scatter plot แสดงกลุ่ม
- Top keywords แต่ละกลุ่ม
- เปรียบเทียบกับหมวดหมู่จริง

In [ ]:
# Google Colab — run this cell first
!pip install pandas scikit-learn plotly openpyxl -q

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
import plotly.graph_objects as go
import plotly.express as px

df = pd.read_excel('std-tisi-b stru.xlsx', skiprows=1)
df.columns = ['เลขที่','ชื่อTH','ประกาศ','ชื่อEN','ขอบข่ายTH','ขอบข่ายEN',
              'วันที่ประกาศ','หมวดหมู่','สถานะ','หน่วยงาน','บังคับ']

# Combine English text for TF-IDF
df['text'] = (df['ขอบข่ายEN'].fillna('') + ' ' + df['ชื่อEN'].fillna('')).str.strip()
df = df[df['text'].str.len() > 10].copy()

print(f'Standards: {len(df)} | Categories: {df["หมวดหมู่"].nunique()}')

## TF-IDF Vectorization

In [ ]:
tfidf = TfidfVectorizer(max_features=300, stop_words='english',
                        ngram_range=(1,2), min_df=2)
X_tfidf = tfidf.fit_transform(df['text'])
feature_names = tfidf.get_feature_names_out()
print(f'TF-IDF matrix: {X_tfidf.shape}')

## Elbow Method + Silhouette Score

In [ ]:
X_dense = X_tfidf.toarray()

inertias = []
sil_scores = []
K_range = range(2, 12)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_dense)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_dense, labels))

fig_elbow = go.Figure()
fig_elbow.add_trace(go.Scatter(x=list(K_range), y=inertias, mode='lines+markers',
    name='Inertia', line=dict(color='#003049', width=2)))
fig_elbow.add_trace(go.Scatter(x=list(K_range), y=sil_scores, mode='lines+markers',
    name='Silhouette', yaxis='y2', line=dict(color='#d62828', width=2)))
fig_elbow.update_layout(
    title='Elbow Method + Silhouette Score',
    xaxis_title='k (จำนวนกลุ่ม)', yaxis_title='Inertia',
    yaxis2=dict(title='Silhouette', overlaying='y', side='right'),
    template='plotly_white', height=400,
    font=dict(family='Sarabun, sans-serif'),
)
fig_elbow.show()

## K-Means Clustering (k=6)

In [ ]:
K = 6
km = KMeans(n_clusters=K, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(X_dense)

sil = silhouette_score(X_dense, df['cluster'])

# Compare with actual categories
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_true = le.fit_transform(df['หมวดหมู่'])
ari = adjusted_rand_score(y_true, df['cluster'])

print(f'Silhouette Score: {sil:.4f}')
print(f'Adjusted Rand Index (vs actual categories): {ari:.4f}')
print(f'\nCluster sizes:')
print(df['cluster'].value_counts().sort_index())

### Chart 1 — PCA Scatter: K-Means กลุ่ม มอก.

ขนาดจุด = คงที่ · สี = กลุ่ม · hover ดูชื่อมาตรฐาน

In [ ]:
CLUSTER_COLORS = ['#003049','#d62828','#f77f00','#2a9d8f','#264653','#e76f51',
                  '#606c38','#e9c46a']

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_dense)
df['PC1'] = coords[:, 0]
df['PC2'] = coords[:, 1]

fig_pca = go.Figure()
for c in sorted(df['cluster'].unique()):
    sub = df[df['cluster']==c]
    fig_pca.add_trace(go.Scatter(
        x=sub['PC1'], y=sub['PC2'], mode='markers',
        name=f'กลุ่ม {c} ({len(sub)})',
        marker=dict(size=10, color=CLUSTER_COLORS[c % len(CLUSTER_COLORS)],
                    opacity=.8, line=dict(width=1, color='white')),
        text=sub['ชื่อTH'].str[:40],
        hovertemplate='<b>%{text}</b><br>หมวด: ' + sub['หมวดหมู่'] + '<br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra>กลุ่ม '+str(c)+'</extra>',
    ))

fig_pca.update_layout(
    title=f'PCA 2D — K-Means {K} กลุ่ม (Silhouette: {sil:.3f}, ARI: {ari:.3f})',
    xaxis_title=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
    yaxis_title=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',
    template='plotly_white', height=550,
    font=dict(family='Sarabun, sans-serif'),
)
fig_pca.show()

### Chart 2 — Top Keywords แต่ละกลุ่ม

In [ ]:
# Get top keywords per cluster from cluster centroids
PALETTE = ['#003049','#d62828','#f77f00','#2a9d8f','#264653','#e76f51']
fig_kw = go.Figure()

for c in range(K):
    centroid = km.cluster_centers_[c]
    top_idx = np.argsort(centroid)[-8:]  # top 8 keywords
    words = [feature_names[i] for i in top_idx]
    scores = centroid[top_idx]
    
    fig_kw.add_trace(go.Bar(
        x=scores, y=words, orientation='h',
        name=f'กลุ่ม {c}',
        marker_color=PALETTE[c % len(PALETTE)],
        hovertemplate=f'กลุ่ม {c}: <b>%{{y}}</b><br>Score: %{{x:.3f}}<extra></extra>',
    ))

fig_kw.update_layout(
    title='Top Keywords per Cluster (TF-IDF Centroid)',
    xaxis_title='TF-IDF Score',
    template='plotly_white', height=600,
    font=dict(family='Sarabun, sans-serif'),
    barmode='group',
    margin=dict(l=180),
)
fig_kw.show()

### Chart 3 — Cluster vs Actual Category

In [ ]:
cross = pd.crosstab(df['cluster'], df['หมวดหมู่'])

fig_cross = go.Figure(go.Heatmap(
    z=cross.values,
    x=cross.columns.tolist(),
    y=[f'กลุ่ม {i}' for i in cross.index],
    colorscale='YlOrRd',
    text=cross.values,
    texttemplate='%{text}',
    hovertemplate='กลุ่ม %{y} × %{x}<br>จำนวน: %{z}<extra></extra>',
))
fig_cross.update_layout(
    title='Cluster × หมวดหมู่จริง (Confusion)',
    xaxis_title='หมวดหมู่จริง', yaxis_title='Cluster',
    template='plotly_white', height=400,
    font=dict(family='Sarabun, sans-serif'),
    xaxis_tickangle=-45,
)
fig_cross.show()

## สรุป

K-Means Clustering จัดกลุ่ม มอก. 148 รายการเป็น 6 กลุ่มจากข้อความขอบข่าย (TF-IDF)

- **Silhouette Score** แสดงคุณภาพการจัดกลุ่ม
- **Adjusted Rand Index** เปรียบเทียบกับหมวดหมู่จริง — ยิ่งสูงยิ่งดี
- Top keywords ชี้ให้เห็นว่าแต่ละกลุ่มเกี่ยวกับผลิตภัณฑ์อะไร